In [0]:
checkpoint = spark.sql("SELECT * FROM de_etl_project.hdb.etl_checkpoint").first()["last_processed_id"]
print(checkpoint)

242905


In [0]:
table = spark.table("de_etl_project.hdb.resale_flat_prices")
print(table.count())

989108


In [0]:
# API to pull data daily
import requests
          
dataset_id = "d_8b84c4ee58e3cfc0ece0d773c8ca6abc"

url = "https://data.gov.sg/api/action/datastore_search?resource_id="  + dataset_id + "&offset=" + str(checkpoint - 1)
        
print(url)
response = requests.get(url)
total_records = response.json()["result"]["total"]
records = response.json()["result"]["records"]
# print(records)
print(total_records, checkpoint)

print(f"records to process: {total_records - checkpoint}")

new_records = []

for offset in range(checkpoint, total_records, 100):
    url = "https://data.gov.sg/api/action/datastore_search?resource_id="  + dataset_id + "&offset=" + str(offset)
    response = requests.get(url)
    data = response.json()

    print(url)
    records = data["result"]["records"]
    new_records.extend(records)

    if "result" not in data:
        print("Failed offset:", offset)
        print("Status code:", response.status_code)
        print("Response:", data)
        break

print(len(new_records))

In [0]:
# create DF
df = spark.createDataFrame(new_records)
display(df)
print(df.count())

In [0]:
from pyspark.sql import functions as f

# fix values for df5 col remaining_lease
# e.g 61 years 04 months -> 61

display(df.head(10))
df = df.withColumn(
    "remaining_lease",
    99 - ( f.year('month') - f.col('lease_commence_date') )
)

display(df.head(10))

In [0]:
# update checkpoint table
new_checkpoint = df.orderBy("_id", ascending=False).first()["_id"]
last_date = df.orderBy("month", ascending=False).first()["month"]

print(new_checkpoint, last_date)

spark.sql(f'''
    UPDATE de_etl_project.hdb.etl_checkpoint
    SET last_processed_id = {new_checkpoint}, date = '{last_date}', last_updated_at = current_date()
    WHERE last_processed_id = {checkpoint}
''')

In [0]:
# make columns match the db table, but exclude record_id

table = spark.table("de_etl_project.hdb.resale_flat_prices")

data_columns = [c for c in table.columns if c != "record_id"]
new_df = df.select(*data_columns)

display(new_df.tail(10))
print(new_df.columns)

In [0]:
# save new records to table
new_df.write.mode("append").saveAsTable("de_etl_project.hdb.resale_flat_prices")

In [0]:
print("Total records:", table.count())